# Path B GT - treino e avaliação em sequência (B1, B3, B4)

Este notebook reutiliza o mesmo pré-processamento do Path B e roda, em sequência, o treino e a avaliação dos classificadores B1, B3 e B4 usando crops GT, com células separadas por path.

Passos cobertos:
- Selecionar detector YOLO;
- Pré-processamento dedicado do Path B (idempotente);
- Treinos em GT crops (B1, B3, B4) em células separadas;
- Avaliações combinadas com e sem TTA+WBF em células separadas;
- Ranqueamento final entre B1, B3 e B4.


In [1]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil
import torch
import pandas as pd
import json

WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'

DATASET_YAML_PATH_B = PROCESSED_DIR / 'dataset_path_B.yaml'

RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A'
MLFLOW_DIR = Path('/root/mlflow')
RESULTS_PATH_B_DIR = WORKSPACE / 'results_path_B'

TRAIN_PATH_B_SCRIPT = TRAIN_DIR / 'train_path_B.py'
EVAL_PATH_B_COMBINED_SCRIPT = EVAL_DIR / 'evaluate_path_B_combined.py'

for p in [MLFLOW_DIR, RESULTS_PATH_B_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_B=', DATASET_YAML_PATH_B)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('RESULTS_PATH_B_DIR =', RESULTS_PATH_B_DIR)
print('TRAIN_PATH_B_SCRIPT=', TRAIN_PATH_B_SCRIPT)
print('EVAL_SCRIPT        =', EVAL_PATH_B_COMBINED_SCRIPT)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)


Python             = /workspace/.venv/bin/python
REPO_ROOT          = /workspace/TrashScan
PROCESSED_DIR      = /workspace/processed_5cls
DATASET_YAML_PATH_B= /workspace/processed_5cls/dataset_path_B.yaml
RUNS_PATH_A_DIR    = /workspace/runs/path_A
RESULTS_PATH_B_DIR = /workspace/results_path_B
TRAIN_PATH_B_SCRIPT= /workspace/TrashScan/train/paths/train_path_B.py
EVAL_SCRIPT        = /workspace/TrashScan/eval/evaluate_path_B_combined.py


In [2]:
if torch.cuda.is_available():
    DEVICE = '0'
    gpu_name = torch.cuda.get_device_properties(0).name
else:
    DEVICE = 'cpu'
    gpu_name = 'cpu'

print('Device:', DEVICE)
print('GPU:', gpu_name)

EPOCHS = 300
BATCH = 8
LR = 5e-5
PATIENCE = 15

EVAL_IMGSZ = 640
EVAL_DET_CONF = 0.001
EVAL_DET_IOU = 0.6

TTA_SCALES = ['512', '640', '768']
TTA_WBF_IOU = '0.55'
TTA_SKIP_BOX_THR = '0.001'

CONFIGS = [
    {
        'key': 'b1',
        'label': 'B1 (resnet50)',
        'classifier': 'resnet50',
        'run_dir': WORKSPACE / 'runs' / 'path_B_gt_b1',
    },
    {
        'key': 'b3',
        'label': 'B3 (vit_b16_imagenet)',
        'classifier': 'vit_b16_imagenet',
        'run_dir': WORKSPACE / 'runs' / 'path_B_gt_b3',
    },
    {
        'key': 'b4',
        'label': 'B4 (vit_l16_imagenet)',
        'classifier': 'vit_l16_imagenet',
        'run_dir': WORKSPACE / 'runs' / 'path_B_gt_b4',
    },
]

B1, B3, B4 = CONFIGS

for cfg in CONFIGS:
    cfg['run_dir'].mkdir(parents=True, exist_ok=True)


Device: 0
GPU: NVIDIA RTX 4000 Ada Generation


## 1) Selecionar detector YOLO

O treino em GT crops não usa o detector para gerar os exemplos. Ainda assim, o script pede `--detector_weights`, e a avaliação combinada precisa do detector.


In [3]:
PREFERRED_DETECTOR = WORKSPACE / 'runs' / 'path_A_5cls' / 'yolov11m_o2o' / 'weights' / 'best.pt'
PATH_A_RUN_DIRS = [
    WORKSPACE / 'runs' / 'path_A_5cls',
    WORKSPACE / 'runs' / 'path_A',
    WORKSPACE / 'runs' / 'path_A_refined_head',
]


def read_yolo_results(run_dir: Path):
    best_pt = run_dir / 'weights' / 'best.pt'
    results_csv = run_dir / 'results.csv'
    metrics_json = run_dir / 'metrics.json'

    if not best_pt.exists():
        return None

    row = {
        'group': run_dir.parent.name,
        'model': run_dir.name,
        'run_dir': run_dir,
        'best_pt': best_pt,
        'mAP50_95': None,
        'mAP50': None,
        'precision': None,
        'recall': None,
        'source': None,
    }

    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]
        map95_col = 'metrics/mAP50-95(B)'
        map50_col = 'metrics/mAP50(B)'
        precision_col = 'metrics/precision(B)'
        recall_col = 'metrics/recall(B)'

        if map95_col in df.columns:
            best_idx = df[map95_col].idxmax()
        elif map50_col in df.columns:
            best_idx = df[map50_col].idxmax()
        else:
            best_idx = df.index[-1]

        best = df.loc[best_idx]
        row['mAP50_95'] = float(best[map95_col]) if map95_col in df.columns else None
        row['mAP50'] = float(best[map50_col]) if map50_col in df.columns else None
        row['precision'] = float(best[precision_col]) if precision_col in df.columns else None
        row['recall'] = float(best[recall_col]) if recall_col in df.columns else None
        row['source'] = 'results.csv'
        return row

    if metrics_json.exists():
        with open(metrics_json, 'r') as f:
            m = json.load(f)
        row['mAP50_95'] = m.get('mAP50_95')
        row['mAP50'] = m.get('mAP50')
        row['precision'] = m.get('precision')
        row['recall'] = m.get('recall')
        row['source'] = 'metrics.json'
        return row

    row['source'] = 'weights_only'
    return row


if PREFERRED_DETECTOR.exists():
    DETECTOR_WEIGHTS = PREFERRED_DETECTOR
    print('Usando detector preferido:', DETECTOR_WEIGHTS)
else:
    records = []
    for base_dir in PATH_A_RUN_DIRS:
        if not base_dir.exists():
            print(f'[warn] Pasta não encontrada: {base_dir}')
            continue
        for run_dir in sorted(base_dir.iterdir()):
            if run_dir.is_dir():
                rec = read_yolo_results(run_dir)
                if rec is not None:
                    records.append(rec)

    df_detectors = pd.DataFrame(records)
    if df_detectors.empty:
        raise FileNotFoundError(
            'Nenhum detector com weights/best.pt foi encontrado em: '
            + ', '.join(str(p) for p in PATH_A_RUN_DIRS)
        )

    df_ranked = df_detectors.copy()
    df_ranked['rank_score'] = df_ranked['mAP50_95'].fillna(df_ranked['mAP50']).fillna(-1)
    df_ranked = df_ranked.sort_values(
        by=['rank_score', 'mAP50', 'precision', 'recall'],
        ascending=False,
        na_position='last',
    ).reset_index(drop=True)

    display(df_ranked[['group', 'model', 'mAP50_95', 'mAP50', 'precision', 'recall', 'source', 'best_pt']])
    DETECTOR_WEIGHTS = Path(df_ranked.iloc[0]['best_pt'])
    print('Melhor detector encontrado:', DETECTOR_WEIGHTS)

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f'Detector não encontrado: {DETECTOR_WEIGHTS}')


Usando detector preferido: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt


## 2) Pré-processamento dedicado do Path B

Esta etapa prepara os dados do Path B a partir do merged_data em `/workspace/processed_5cls/merged_data`,
gerando `images`, `labels`, `crops` e `dataset_path_B.yaml` dentro do root `/workspace/processed_5cls`.

A célula abaixo é idempotente: se a estrutura `train/val/test/path_B/{images,labels,crops}` e o YAML já existirem, ela só reutiliza o dataset. Para recriar o pré-processamento manualmente, altere `FORCE_PREPROCESS_PATH_B` para `True`.


In [4]:
# Pré-processamento do Path B a partir do merged_data. Rode uma vez; os outros treinos GT reutilizam o mesmo root.
FORCE_PREPROCESS_PATH_B = False

PREPROCESS_PATH_B_SCRIPT = DATA_DIR / 'preprocess.py'
MERGED_ROOT = PROCESSED_DIR / 'merged_data'
REQUIRED_PATH_B_ITEMS = [DATASET_YAML_PATH_B]

for split in ['train', 'val', 'test']:
    for subdir in ['images', 'labels', 'crops']:
        REQUIRED_PATH_B_ITEMS.append(PROCESSED_DIR / split / 'path_B' / subdir)

missing_path_b_items = [p for p in REQUIRED_PATH_B_ITEMS if not p.exists()]

if FORCE_PREPROCESS_PATH_B or missing_path_b_items:
    if FORCE_PREPROCESS_PATH_B:
        print('FORCE_PREPROCESS_PATH_B=True; executando pré-processamento do Path B.')
    else:
        print('Pré-processamento do Path B ausente ou incompleto. Itens faltantes:')
        for p in missing_path_b_items:
            print(' -', p)

    if not MERGED_ROOT.exists():
        raise FileNotFoundError(f'merged_data não encontrado: {MERGED_ROOT}')
    if not PREPROCESS_PATH_B_SCRIPT.exists():
        raise FileNotFoundError(f'Script de pré-processamento não encontrado: {PREPROCESS_PATH_B_SCRIPT}')

    run_cmd([
        sys.executable, str(PREPROCESS_PATH_B_SCRIPT),
        '--taco_root', str(MERGED_ROOT),
        '--output_root', str(PROCESSED_DIR),
        '--path', 'B',
    ])
else:
    print('Pré-processamento do Path B já encontrado:', PROCESSED_DIR)


Pré-processamento do Path B já encontrado: /workspace/processed_5cls


## 3) Conferir crops GT

Este notebook treina em `CropDataset`, então ele depende de `train/val/test/path_B/crops/{class_idx}`.


In [5]:
for split in ['train', 'val', 'test']:
    crop_root = PROCESSED_DIR / split / 'path_B' / 'crops'
    if not crop_root.exists():
        raise FileNotFoundError(f'Crops GT não encontrados: {crop_root}')

    counts = {}
    for cls_dir in sorted(crop_root.iterdir()):
        if cls_dir.is_dir():
            counts[cls_dir.name] = len(list(cls_dir.glob('*.jpg')))

    print(split, crop_root)
    print('  total:', sum(counts.values()), 'por classe:', counts)


train /workspace/processed_5cls/train/path_B/crops
  total: 15678 por classe: {'0': 6025, '1': 1307, '2': 1245, '3': 500, '4': 6601}
val /workspace/processed_5cls/val/path_B/crops
  total: 3447 por classe: {'0': 1411, '1': 267, '2': 282, '3': 107, '4': 1380}
test /workspace/processed_5cls/test/path_B/crops
  total: 3208 por classe: {'0': 1312, '1': 280, '2': 265, '3': 88, '4': 1263}


## 4) Treinos em GT crops (B1, B3, B4)

Sem `--use_yolo_crops`. O classificador é treinado nos crops GT pré-gerados.
Parâmetros fixos: epochs=300, batch=8, patience=15.


In [16]:
print('---', B1['label'], '---')
print('Parâmetros de treino:')
print('Detector weights:', DETECTOR_WEIGHTS)
print('Crops dir:', PROCESSED_DIR)
print('Output dir:', B1['run_dir'])
print('Classifier:', B1['classifier'])
print('Epochs:', EPOCHS)
print('Batch:', BATCH)
print('Learning rate:', LR)
print('Patience:', PATIENCE)
print('Device:', DEVICE)
print('Use YOLO crops:', False)

run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B1['run_dir']),
    '--classifiers', B1['classifier'],
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--lr', str(LR),
    '--patience', str(PATIENCE),
    '--device', str(DEVICE),
])


--- B1 (resnet50) ---
Parâmetros de treino:
Detector weights: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
Crops dir: /workspace/processed_5cls
Output dir: /workspace/runs/path_B_gt_b1
Classifier: resnet50
Epochs: 300
Batch: 8
Learning rate: 5e-05
Patience: 15
Device: 0
Use YOLO crops: False
$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_gt_b1 --classifiers resnet50 --epochs 300 --batch 8 --lr 5e-05 --patience 15 --device 0
Device : cuda:0
GPU    : NVIDIA RTX 4000 Ada Generation  VRAM: 21.1GB
Class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

  Mode: loading pre-generated GT crops from disk
  [train] 15678 GT crops across 5 classes
  [val] 3447 GT crops across 5 classes
  [test] 3208 GT crops across 5 classes

──────────────────────────────────────────

  Built resnet50 (resnet50.a1_in1k)  pretrained=True  23.5M params


  Ep   1/300 | train loss=1.3451 acc=0.5691 | val loss=1.0583 acc=0.6194 f1=0.2738


  Ep   2/300 | train loss=1.1877 acc=0.6309 | val loss=0.9248 acc=0.6916 f1=0.4729


  Ep   3/300 | train loss=1.0653 acc=0.6551 | val loss=0.8465 acc=0.6951 f1=0.5355


...

  Ep  95/300 | train loss=0.0785 acc=0.9701 | val loss=0.7658 acc=0.8564 f1=0.8382


  Ep  96/300 | train loss=0.0848 acc=0.9700 | val loss=0.8107 acc=0.8573 f1=0.8367
  Early stopping at epoch 96 (best epoch 81, val_acc=0.8622)


  Test: 100%|██████████| 401/401 [00:08<00:00, 46.46it/s]



  [resnet50]  accuracy=0.8809  f1=0.8524  latency=7.00ms

  [resnet50]  best_epoch=81  val_acc=0.8622  test_acc=0.8809  f1=0.8524

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
            accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                    
resnet50      0.8809     0.8352  0.8738 0.8524      7.0020      0.9574    0.8884    0.8766    0.8629    0.9555


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B_gt_b1', '--classifiers', 'resnet50', '--epochs', '300', '--batch', '8', '--lr', '5e-05', '--patience', '15', '--device', '0'], returncode=0)

In [11]:
print('---', B3['label'], '---')
print('Parâmetros de treino:')
print('Detector weights:', DETECTOR_WEIGHTS)
print('Crops dir:', PROCESSED_DIR)
print('Output dir:', B3['run_dir'])
print('Classifier:', B3['classifier'])
print('Epochs:', EPOCHS)
print('Batch:', BATCH)
print('Learning rate:', LR)
print('Patience:', PATIENCE)
print('Device:', DEVICE)
print('Use YOLO crops:', False)

run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B3['run_dir']),
    '--classifiers', B3['classifier'],
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--lr', str(LR),
    '--patience', str(PATIENCE),
    '--device', str(DEVICE),
])


--- B3 (vit_b16_imagenet) ---
Parâmetros de treino:
Detector weights: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
Crops dir: /workspace/processed_5cls
Output dir: /workspace/runs/path_B_gt_b3
Classifier: vit_b16_imagenet
Epochs: 300
Batch: 8
Learning rate: 5e-05
Patience: 15
Device: 0
Use YOLO crops: False
$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_gt_b3 --classifiers vit_b16_imagenet --epochs 300 --batch 8 --lr 5e-05 --patience 15 --device 0


Device : cuda:0
GPU    : NVIDIA RTX 4000 Ada Generation  VRAM: 21.1GB
Class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

  Mode: loading pre-generated GT crops from disk
  [train] 15678 GT crops across 5 classes
  [val] 3447 GT crops across 5 classes
  [test] 3208 GT crops across 5 classes

────────────────────────────────────────────────────────────
  Classifier : vit_b16_imagenet
────────────────────────────────────────────────────────────
  Built vit_b16_imagenet (vit_base_patch16_224.augreg_in21k_ft_in1k)  pretrained=True  85.8M params


  Ep   1/300 | train loss=0.9638 acc=0.6896 | val loss=0.7036 acc=0.7325 f1=0.6394


  Ep   2/300 | train loss=0.6953 acc=0.7685 | val loss=0.5827 acc=0.7818 f1=0.7187


  Ep   3/300 | train loss=0.5879 acc=0.7991 | val loss=0.5843 acc=0.7961 f1=0.7141


...

  Ep  80/300 | train loss=0.0489 acc=0.9814 | val loss=0.6638 acc=0.8776 f1=0.8594


  Ep  81/300 | train loss=0.0488 acc=0.9830 | val loss=0.6388 acc=0.8726 f1=0.8441
  Early stopping at epoch 81 (best epoch 66, val_acc=0.8877)


  Test: 100%|██████████| 401/401 [00:12<00:00, 31.48it/s]



  [vit_b16_imagenet]  accuracy=0.8937  f1=0.8825  latency=5.22ms

  [vit_b16_imagenet]  best_epoch=66  val_acc=0.8877  test_acc=0.8937  f1=0.8825

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_b16_imagenet    0.8937     0.8792  0.8871 0.8825      5.2200      0.9609    0.9277    0.9279    0.8893    0.9651


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B_gt_b3', '--classifiers', 'vit_b16_imagenet', '--epochs', '300', '--batch', '8', '--lr', '5e-05', '--patience', '15', '--device', '0'], returncode=0)

In [8]:
print('---', B4['label'], '---')
print('Parâmetros de treino:')
print('Detector weights:', DETECTOR_WEIGHTS)
print('Crops dir:', PROCESSED_DIR)
print('Output dir:', B4['run_dir'])
print('Classifier:', B4['classifier'])
print('Epochs:', EPOCHS)
print('Batch:', BATCH)
print('Learning rate:', LR)
print('Patience:', PATIENCE)
print('Device:', DEVICE)
print('Use YOLO crops:', False)

run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B4['run_dir']),
    '--classifiers', B4['classifier'],
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--lr', str(LR),
    '--patience', str(PATIENCE),
    '--device', str(DEVICE),
])


--- B4 (vit_l16_imagenet) ---
Parâmetros de treino:
Detector weights: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
Crops dir: /workspace/processed_5cls
Output dir: /workspace/runs/path_B_gt_b4
Classifier: vit_l16_imagenet
Epochs: 300
Batch: 8
Learning rate: 5e-05
Patience: 15
Device: 0
Use YOLO crops: False
TTA+WBF no treino: False
$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_gt_b4 --classifiers vit_l16_imagenet --epochs 300 --batch 8 --lr 5e-05 --patience 15 --device 0


Device : cuda:0
GPU    : NVIDIA RTX 4000 Ada Generation  VRAM: 21.0GB
Class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

  Mode: loading pre-generated GT crops from disk
  [train] 15678 GT crops across 5 classes
  [val] 3447 GT crops across 5 classes
  [test] 3208 GT crops across 5 classes

────────────────────────────────────────────────────────────
  Classifier : vit_l16_imagenet
────────────────────────────────────────────────────────────


  Built vit_l16_imagenet (vit_large_patch16_224.augreg_in21k_ft_in1k)  pretrained=True  303.3M params


  Ep   1/300 | train loss=0.8311 acc=0.7282 | val loss=0.6572 acc=0.7514 f1=0.6987


  Ep   2/300 | train loss=0.5751 acc=0.8046 | val loss=0.5542 acc=0.8001 f1=0.7461


...

  Ep  63/300 | train loss=0.0550 acc=0.9824 | val loss=0.5349 acc=0.8750 f1=0.8645


  Ep  64/300 | train loss=0.0536 acc=0.9823 | val loss=0.6387 acc=0.8726 f1=0.8636
  Early stopping at epoch 64 (best epoch 49, val_acc=0.8848)


  Test: 100%|██████████| 401/401 [00:39<00:00, 10.03it/s]



  [vit_l16_imagenet]  accuracy=0.8940  f1=0.8819  latency=17.38ms

  [vit_l16_imagenet]  best_epoch=49  val_acc=0.8848  test_acc=0.8940  f1=0.8819

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_l16_imagenet    0.8940     0.8733  0.8923 0.8819     17.3840      0.9565    0.9201    0.9315    0.8751    0.9676


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B_gt_b4', '--classifiers', 'vit_l16_imagenet', '--epochs', '300', '--batch', '8', '--lr', '5e-05', '--patience', '15', '--device', '0'], returncode=0)

## 5) Resumo do treino (B1, B3, B4)


In [7]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B1['run_dir']),
    '--summarize',
])


$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_gt_b1 --summarize

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
            accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                    
resnet50      0.8809     0.8352  0.8738 0.8524      7.0020      0.9574    0.8884    0.8766    0.8629    0.9555


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B_gt_b1', '--summarize'], returncode=0)

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B3['run_dir']),
    '--summarize',
])


$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_gt_b3 --summarize

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_b16_imagenet    0.8937     0.8792  0.8871 0.8825      5.2200      0.9609    0.9277    0.9279    0.8893    0.9651


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B_gt_b3', '--summarize'], returncode=0)

In [9]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B4['run_dir']),
    '--summarize',
])


$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_gt_b4 --summarize



PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_l16_imagenet    0.8940     0.8733  0.8923 0.8819     17.3840      0.9565    0.9201    0.9315    0.8751    0.9676


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B_gt_b4', '--summarize'], returncode=0)

## 6) Avaliação combinada sem TTA+WBF (B1, B3, B4)

Esta seção avalia detector + classificador no modo standard.


In [9]:
CLASSIFIER_DIR = B1['run_dir']
OUTPUT_DIR_STANDARD = RESULTS_PATH_B_DIR / 'b1_5cls_gt_standard'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B1['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_STANDARD),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
]

run_cmd(cmd)


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_path_B_combined.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --classifier_dir /workspace/runs/path_B_gt_b1 --classifiers resnet50 --data_yaml /workspace/processed_5cls/dataset_path_B.yaml --output /workspace/results_path_B/b1_5cls_gt_standard --device 0 --imgsz 640 --det_conf 0.001 --det_iou 0.6


Device: cuda:0
Test set: /workspace/processed_5cls/test/path_B/images

Loading detector: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt

────────────────────────────────────────────────────────────
  Evaluating: resnet50
────────────────────────────────────────────────────────────
  Loaded resnet50: 23.5M params from best.pt

  Running combined inference on 1392 test images...
  TTA+WBF: False


  resnet50: 100%|██████████| 1392/1392 [02:52<00:00,  8.07it/s]



  [resnet50]  mAP50=0.6775  mAP50-95=0.4574  fps=8.1
    AP50 plastic : 0.7014  (n_gt=1313)
    AP50 paper   : 0.6723  (n_gt=280)
    AP50 metal   : 0.6084  (n_gt=265)
    AP50 glass   : 0.7054  (n_gt=88)
    AP50 other   : 0.7003  (n_gt=1263)
  Saved: /workspace/results_path_B/b1_5cls_gt_standard/individual/B_resnet50_combined.json

PATH B COMBINED — FINAL COMPARISON
Classifier                  mAP50  mAP50-95    FPS
──────────────────────────────────────────────────
  resnet50                 0.6775    0.4574    8.1

Summary saved: /workspace/results_path_B/b1_5cls_gt_standard/path_B_combined_summary.json


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_path_B_combined.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--classifier_dir', '/workspace/runs/path_B_gt_b1', '--classifiers', 'resnet50', '--data_yaml', '/workspace/processed_5cls/dataset_path_B.yaml', '--output', '/workspace/results_path_B/b1_5cls_gt_standard', '--device', '0', '--imgsz', '640', '--det_conf', '0.001', '--det_iou', '0.6'], returncode=0)

In [13]:
CLASSIFIER_DIR = B3['run_dir']
OUTPUT_DIR_STANDARD = RESULTS_PATH_B_DIR / 'b3_5cls_gt_standard'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B3['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_STANDARD),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
]

run_cmd(cmd)


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_path_B_combined.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --classifier_dir /workspace/runs/path_B_gt_b3 --classifiers vit_b16_imagenet --data_yaml /workspace/processed_5cls/dataset_path_B.yaml --output /workspace/results_path_B/b3_5cls_gt_standard --device 0 --imgsz 640 --det_conf 0.001 --det_iou 0.6
Device: cuda:0
Test set: /workspace/processed_5cls/test/path_B/images

Loading detector: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt

────────────────────────────────────────────────────────────
  Evaluating: vit_b16_imagenet
────────────────────────────────────────────────────────────
  Loaded vit_b16_imagenet: 85.8M params from best.pt

  Running combined inference on 1392 test images...
  TTA+WBF: False


  vit_b16_imagenet: 100%|██████████| 1392/1392 [02:56<00:00,  7.90it/s]



  [vit_b16_imagenet]  mAP50=0.7097  mAP50-95=0.4768  fps=7.9
    AP50 plastic : 0.7082  (n_gt=1313)
    AP50 paper   : 0.7268  (n_gt=280)
    AP50 metal   : 0.6595  (n_gt=265)
    AP50 glass   : 0.7465  (n_gt=88)
    AP50 other   : 0.7076  (n_gt=1263)
  Saved: /workspace/results_path_B/b3_5cls_gt_standard/individual/B_vit_b16_imagenet_combined.json

PATH B COMBINED — FINAL COMPARISON
Classifier                  mAP50  mAP50-95    FPS
──────────────────────────────────────────────────
  vit_b16_imagenet         0.7097    0.4768    7.9

Summary saved: /workspace/results_path_B/b3_5cls_gt_standard/path_B_combined_summary.json


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_path_B_combined.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--classifier_dir', '/workspace/runs/path_B_gt_b3', '--classifiers', 'vit_b16_imagenet', '--data_yaml', '/workspace/processed_5cls/dataset_path_B.yaml', '--output', '/workspace/results_path_B/b3_5cls_gt_standard', '--device', '0', '--imgsz', '640', '--det_conf', '0.001', '--det_iou', '0.6'], returncode=0)

In [5]:
CLASSIFIER_DIR = B4['run_dir']
OUTPUT_DIR_STANDARD = RESULTS_PATH_B_DIR / 'b4_5cls_gt_standard'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B4['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_STANDARD),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
]

run_cmd(cmd)


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_path_B_combined.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --classifier_dir /workspace/runs/path_B_gt_b4 --classifiers vit_l16_imagenet --data_yaml /workspace/processed_5cls/dataset_path_B.yaml --output /workspace/results_path_B/b4_5cls_gt_standard --device 0 --imgsz 640 --det_conf 0.001 --det_iou 0.6


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cuda:0
Test set: /workspace/processed_5cls/test/path_B/images

Loading detector: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt

────────────────────────────────────────────────────────────
  Evaluating: vit_l16_imagenet
────────────────────────────────────────────────────────────
  Loaded vit_l16_imagenet: 303.3M params from best.pt

  Running combined inference on 1392 test images...
  TTA+WBF: False


  vit_l16_imagenet: 100%|██████████| 1392/1392 [05:19<00:00,  4.36it/s]



  [vit_l16_imagenet]  mAP50=0.7039  mAP50-95=0.4740  fps=4.4
    AP50 plastic : 0.7124  (n_gt=1313)
    AP50 paper   : 0.7246  (n_gt=280)
    AP50 metal   : 0.6418  (n_gt=265)
    AP50 glass   : 0.7350  (n_gt=88)
    AP50 other   : 0.7054  (n_gt=1263)
  Saved: /workspace/results_path_B/b4_5cls_gt_standard/individual/B_vit_l16_imagenet_combined.json

PATH B COMBINED — FINAL COMPARISON
Classifier                  mAP50  mAP50-95    FPS
──────────────────────────────────────────────────
  vit_l16_imagenet         0.7039    0.4740    4.4

Summary saved: /workspace/results_path_B/b4_5cls_gt_standard/path_B_combined_summary.json


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_path_B_combined.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--classifier_dir', '/workspace/runs/path_B_gt_b4', '--classifiers', 'vit_l16_imagenet', '--data_yaml', '/workspace/processed_5cls/dataset_path_B.yaml', '--output', '/workspace/results_path_B/b4_5cls_gt_standard', '--device', '0', '--imgsz', '640', '--det_conf', '0.001', '--det_iou', '0.6'], returncode=0)

## 7) Avaliação combinada com TTA+WBF (B1, B3, B4)

Esta seção avalia o mesmo classificador treinado em GT crops, mas com o detector usando TTA multi-scale + WBF.


In [10]:
CLASSIFIER_DIR = B1['run_dir']
OUTPUT_DIR_TTA_WBF = RESULTS_PATH_B_DIR / 'b1_5cls_gt_tta_wbf'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B1['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_TTA_WBF),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
    '--use_tta_wbf',
    '--tta_scales', *TTA_SCALES,
    '--tta_flip',
    '--tta_wbf_iou', TTA_WBF_IOU,
    '--tta_skip_box_thr', TTA_SKIP_BOX_THR,
]

run_cmd(cmd)


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_path_B_combined.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --classifier_dir /workspace/runs/path_B_gt_b1 --classifiers resnet50 --data_yaml /workspace/processed_5cls/dataset_path_B.yaml --output /workspace/results_path_B/b1_5cls_gt_tta_wbf --device 0 --imgsz 640 --det_conf 0.001 --det_iou 0.6 --use_tta_wbf --tta_scales 512 640 768 --tta_flip --tta_wbf_iou 0.55 --tta_skip_box_thr 0.001
Device: cuda:0
Test set: /workspace/processed_5cls/test/path_B/images

Loading detector: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt

────────────────────────────────────────────────────────────
  Evaluating: resnet50
────────────────────────────────────────────────────────────
  Loaded resnet50: 23.5M params from best.pt

  Running combined inference on 1392 test images...
  TTA+WBF: True
  TTA scales: [512, 640, 768]
  TTA flip: True
  TTA WBF IoU: 0.55
  TTA skip box thr: 0.001


  resnet50: 100%|██████████| 1392/1392 [04:18<00:00,  5.38it/s]



  [resnet50]  mAP50=0.6907  mAP50-95=0.4738  fps=5.4
    AP50 plastic : 0.7164  (n_gt=1313)
    AP50 paper   : 0.7085  (n_gt=280)
    AP50 metal   : 0.6357  (n_gt=265)
    AP50 glass   : 0.6833  (n_gt=88)
    AP50 other   : 0.7096  (n_gt=1263)
  Saved: /workspace/results_path_B/b1_5cls_gt_tta_wbf/individual/B_resnet50_combined.json

PATH B COMBINED — FINAL COMPARISON
Classifier                  mAP50  mAP50-95    FPS
──────────────────────────────────────────────────
  resnet50                 0.6907    0.4738    5.4

Summary saved: /workspace/results_path_B/b1_5cls_gt_tta_wbf/path_B_combined_summary.json


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_path_B_combined.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--classifier_dir', '/workspace/runs/path_B_gt_b1', '--classifiers', 'resnet50', '--data_yaml', '/workspace/processed_5cls/dataset_path_B.yaml', '--output', '/workspace/results_path_B/b1_5cls_gt_tta_wbf', '--device', '0', '--imgsz', '640', '--det_conf', '0.001', '--det_iou', '0.6', '--use_tta_wbf', '--tta_scales', '512', '640', '768', '--tta_flip', '--tta_wbf_iou', '0.55', '--tta_skip_box_thr', '0.001'], returncode=0)

In [14]:
CLASSIFIER_DIR = B3['run_dir']
OUTPUT_DIR_TTA_WBF = RESULTS_PATH_B_DIR / 'b3_5cls_gt_tta_wbf'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B3['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_TTA_WBF),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
    '--use_tta_wbf',
    '--tta_scales', *TTA_SCALES,
    '--tta_flip',
    '--tta_wbf_iou', TTA_WBF_IOU,
    '--tta_skip_box_thr', TTA_SKIP_BOX_THR,
]

run_cmd(cmd)


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_path_B_combined.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --classifier_dir /workspace/runs/path_B_gt_b3 --classifiers vit_b16_imagenet --data_yaml /workspace/processed_5cls/dataset_path_B.yaml --output /workspace/results_path_B/b3_5cls_gt_tta_wbf --device 0 --imgsz 640 --det_conf 0.001 --det_iou 0.6 --use_tta_wbf --tta_scales 512 640 768 --tta_flip --tta_wbf_iou 0.55 --tta_skip_box_thr 0.001
Device: cuda:0
Test set: /workspace/processed_5cls/test/path_B/images

Loading detector: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt

────────────────────────────────────────────────────────────
  Evaluating: vit_b16_imagenet
────────────────────────────────────────────────────────────
  Loaded vit_b16_imagenet: 85.8M params from best.pt

  Running combined inference on 1392 test images...
  TTA+WBF: True
  TTA scales: [512, 640, 768]
  TTA flip: True
  TTA WBF IoU: 0.55
  TTA skip box

  vit_b16_imagenet: 100%|██████████| 1392/1392 [04:21<00:00,  5.33it/s]



  [vit_b16_imagenet]  mAP50=0.7121  mAP50-95=0.4867  fps=5.3
    AP50 plastic : 0.7236  (n_gt=1313)
    AP50 paper   : 0.7459  (n_gt=280)
    AP50 metal   : 0.6610  (n_gt=265)
    AP50 glass   : 0.7185  (n_gt=88)
    AP50 other   : 0.7113  (n_gt=1263)
  Saved: /workspace/results_path_B/b3_5cls_gt_tta_wbf/individual/B_vit_b16_imagenet_combined.json

PATH B COMBINED — FINAL COMPARISON
Classifier                  mAP50  mAP50-95    FPS
──────────────────────────────────────────────────
  vit_b16_imagenet         0.7121    0.4867    5.3

Summary saved: /workspace/results_path_B/b3_5cls_gt_tta_wbf/path_B_combined_summary.json


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_path_B_combined.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--classifier_dir', '/workspace/runs/path_B_gt_b3', '--classifiers', 'vit_b16_imagenet', '--data_yaml', '/workspace/processed_5cls/dataset_path_B.yaml', '--output', '/workspace/results_path_B/b3_5cls_gt_tta_wbf', '--device', '0', '--imgsz', '640', '--det_conf', '0.001', '--det_iou', '0.6', '--use_tta_wbf', '--tta_scales', '512', '640', '768', '--tta_flip', '--tta_wbf_iou', '0.55', '--tta_skip_box_thr', '0.001'], returncode=0)

In [6]:
CLASSIFIER_DIR = B4['run_dir']
OUTPUT_DIR_TTA_WBF = RESULTS_PATH_B_DIR / 'b4_5cls_gt_tta_wbf'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B4['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_TTA_WBF),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
    '--use_tta_wbf',
    '--tta_scales', *TTA_SCALES,
    '--tta_flip',
    '--tta_wbf_iou', TTA_WBF_IOU,
    '--tta_skip_box_thr', TTA_SKIP_BOX_THR,
]

run_cmd(cmd)


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_path_B_combined.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --classifier_dir /workspace/runs/path_B_gt_b4 --classifiers vit_l16_imagenet --data_yaml /workspace/processed_5cls/dataset_path_B.yaml --output /workspace/results_path_B/b4_5cls_gt_tta_wbf --device 0 --imgsz 640 --det_conf 0.001 --det_iou 0.6 --use_tta_wbf --tta_scales 512 640 768 --tta_flip --tta_wbf_iou 0.55 --tta_skip_box_thr 0.001
Device: cuda:0
Test set: /workspace/processed_5cls/test/path_B/images

Loading detector: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt

────────────────────────────────────────────────────────────
  Evaluating: vit_l16_imagenet
────────────────────────────────────────────────────────────
  Loaded vit_l16_imagenet: 303.3M params from best.pt

  Running combined inference on 1392 test images...
  TTA+WBF: True
  TTA scales: [512, 640, 768]
  TTA flip: True
  TTA WBF IoU: 0.55
  TTA skip bo

  vit_l16_imagenet: 100%|██████████| 1392/1392 [06:21<00:00,  3.65it/s]



  [vit_l16_imagenet]  mAP50=0.7112  mAP50-95=0.4857  fps=3.7
    AP50 plastic : 0.7277  (n_gt=1313)
    AP50 paper   : 0.7496  (n_gt=280)
    AP50 metal   : 0.6606  (n_gt=265)
    AP50 glass   : 0.7063  (n_gt=88)
    AP50 other   : 0.7118  (n_gt=1263)
  Saved: /workspace/results_path_B/b4_5cls_gt_tta_wbf/individual/B_vit_l16_imagenet_combined.json

PATH B COMBINED — FINAL COMPARISON
Classifier                  mAP50  mAP50-95    FPS
──────────────────────────────────────────────────
  vit_l16_imagenet         0.7112    0.4857    3.6

Summary saved: /workspace/results_path_B/b4_5cls_gt_tta_wbf/path_B_combined_summary.json


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_path_B_combined.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--classifier_dir', '/workspace/runs/path_B_gt_b4', '--classifiers', 'vit_l16_imagenet', '--data_yaml', '/workspace/processed_5cls/dataset_path_B.yaml', '--output', '/workspace/results_path_B/b4_5cls_gt_tta_wbf', '--device', '0', '--imgsz', '640', '--det_conf', '0.001', '--det_iou', '0.6', '--use_tta_wbf', '--tta_scales', '512', '640', '768', '--tta_flip', '--tta_wbf_iou', '0.55', '--tta_skip_box_thr', '0.001'], returncode=0)

## 8) Ranqueamento entre B1, B3 e B4


In [7]:
rows = []

def add_summary_rows(summary_path: Path, model_label: str, eval_mode: str):
    if not summary_path.exists():
        print(f'[warn] Resultado não encontrado para {model_label} ({eval_mode}): {summary_path}')
        return
    data = json.loads(summary_path.read_text())
    if isinstance(data, list):
        for row in data:
            row = dict(row)
            row['model'] = model_label
            row['eval_mode'] = eval_mode
            rows.append(row)
    else:
        row = dict(data)
        row['model'] = model_label
        row['eval_mode'] = eval_mode
        rows.append(row)

for cfg in CONFIGS:
    add_summary_rows(
        RESULTS_PATH_B_DIR / (cfg['key'] + '_5cls_gt_standard') / 'path_B_combined_summary.json',
        cfg['label'],
        'standard',
    )
    add_summary_rows(
        RESULTS_PATH_B_DIR / (cfg['key'] + '_5cls_gt_tta_wbf') / 'path_B_combined_summary.json',
        cfg['label'],
        'tta_wbf',
    )

if not rows:
    print('Nenhum resumo encontrado ainda.')
else:
    df = pd.DataFrame(rows)
    score_col = 'mAP50'
    cols = [c for c in ['model', 'eval_mode', 'classifier', score_col, 'mAP50', 'mAP50_95', 'fps', 'inference_mode'] if c in df.columns]

    if score_col not in df.columns:
        print('[warn] Coluna mAP50 não encontrada; não é possível ranquear.')
    else:
        for mode in ['standard', 'tta_wbf']:
            df_mode = df[df['eval_mode'] == mode].copy()
            if df_mode.empty:
                print(f'[warn] Sem dados para modo: {mode}')
                continue
            df_mode = df_mode.sort_values(by=[score_col], ascending=False).reset_index(drop=True)
            print(f'Ranking ({mode}) - ordenado por {score_col}')
            display(df_mode[cols])


Ranking (standard) - ordenado por mAP50


,model,eval_mode,classifier,mAP50,mAP50,mAP50_95,fps,inference_mode
0,B3 (vit_b16_imagenet),standard,vit_b16_imagenet,0.70972,0.70972,0.47684,7.90,standard
1,B4 (vit_l16_imagenet),standard,vit_l16_imagenet,0.70386,0.70386,0.47398,4.36,standard
2,B1 (resnet50),standard,resnet50,0.67755,0.67755,0.45743,8.07,standard


Ranking (tta_wbf) - ordenado por mAP50


,model,eval_mode,classifier,mAP50,mAP50,mAP50_95,fps,inference_mode
0,B3 (vit_b16_imagenet),tta_wbf,vit_b16_imagenet,0.71208,0.71208,0.48674,5.33,tta_wbf
1,B4 (vit_l16_imagenet),tta_wbf,vit_l16_imagenet,0.71121,0.71121,0.48569,3.65,tta_wbf
2,B1 (resnet50),tta_wbf,resnet50,0.69069,0.69069,0.47384,5.38,tta_wbf
